# 21 Clustering V2 — 단지 태그 · 분석용 군집 · 유사 단지

## 목적

21은 세 가지 결과를 만들고, 쓰임마다 맞는 도구를 쓴다.

| 쓰임 | 필요한 성질 | 도구 |
|---|---|---|
| 웹 단지 태그 ("대단지 · 신축 · 역 출입구 250m 이내") | 이유를 설명할 수 있고, 재학습해도 바뀌지 않음 | **규칙 태그** (법령 · 공식 기준 · 관행 기준을 출처와 함께) |
| 시장 구조 분석 · 11 회귀 Ablation | 가격 미사용, 거래 시점 배정 | **KMeans** (입지 · 교육 축) |
| 유사 단지 | 목록마다 목적이 분명함 | **KNN** (같은 구 비교 / 서울 전체 대안) |

## 이렇게 바꾼 이유 (첫 버전의 검토 결과)

첫 버전은 KMeans 유형을 웹 태그로 쓰려 했다. 다시 검토하니 문제가 있었다 (2장에서 재현).

- 학습 단지 7,581개 중 46%가 50세대 미만 · 65%가 1개 동 단지이고, 150세대 이상 단지(33%)가 거래의 85%를 차지한다.
  전체 단지로 학습한 주택특성 군집의 Silhouette 0.351은 작은 단지가 만든 값이었고, **150세대 이상 단지 안에서는 0.09**였다.
- 학습 모집단을 바꾸거나(분리 · 세대수 가중) Feature를 바꿔도 주요 단지 Silhouette은 0.25 안팎이고,
  HDBSCAN · GMM도 자연스러운 무리를 찾지 못했다. 서울 단지 특성은 **연속 분포**라 군집 경계는 임의적이다.
- 그래서 사용자에게 보이는 태그는 규칙으로 만들고, KMeans는 분석 · Ablation용으로만 둔다.

## 원칙

- 학습 단지는 2025-12까지 거래된 단지 7,581개 (V1과 동일), 2026 신규 단지 290개는 같은 규칙 · 모델로 배정한다.
- 가격(거래금액 · 평당가 · 공시가격)은 태그 · 군집 · 유사도에 쓰지 않는다. 사후 해석과 검증에만 쓴다.
- 시점: 학습은 `_학습기준`(2025-07-01), 웹 표시는 `_최신`(2026-07-01). 열 이름을 `v2_trade`와 맞춰 11에서 거래 시점으로 배정한다.
- 평가는 사용자가 주로 보는 **주요 단지(150세대 이상)** 기준과 거래 가중을 함께 본다.

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import HDBSCAN, KMeans
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_samples, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import BallTree, NearestNeighbors
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data").exists() and (path / ".env").exists():
            return path
    raise FileNotFoundError("프로젝트 루트를 찾을 수 없습니다.")


PROJECT_DIR = find_project_root()
PROCESSED_DIR = PROJECT_DIR / "data" / "v2_5year_seoul" / "processed"
V1_MODEL_DIR = PROJECT_DIR / "models" / "clustering" / "v1"
MODEL_DIR = PROJECT_DIR / "models" / "clustering" / "v2"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_END = pd.Timestamp("2026-01-01")  # V1과 같은 학습 구간: 2025-12 거래까지
SNAP_YEAR = {"학습기준": 2025, "최신": 2026}  # 연식 계산 기준연도 (스냅샷 2025-07-01 / 2026-07-01)
MAIN_MIN_HOUSEHOLDS = 150  # 주요 단지 = 의무관리 대상이 될 수 있는 규모 (공동주택관리법 제2조제1항제2호)
SEED = 42

cx = pd.read_parquet(PROCESSED_DIR / "v2_complex.parquet")
trades = pd.read_parquet(PROCESSED_DIR / "df_final_v1.parquet",
                         columns=["단지일련번호", "거래월일자", "전용면적", "층", "건축년도",
                                  "역세권거리_m", "주요일자리거리_m", "평당가"])
trades["거래월일자"] = pd.to_datetime(trades["거래월일자"])
cx["학습단지"] = cx["첫거래월"] < TRAIN_END
print("단지 테이블:", cx.shape, "| 학습 단지", f"{cx['학습단지'].sum():,}", "| 2026 신규 단지", f"{(~cx['학습단지']).sum():,}")

단지 테이블: (7871, 81) | 학습 단지 7,581 | 2026 신규 단지 290


## 1. 단지 테이블 준비

- 거래 기반 값(학습 구간 거래의 단지별 중앙값)은 V1 재현 · 05b 결측 보정 · 사후 가격 해석 · 거래 가중에만 쓴다.
- 05b 결측(건축물대장 · 공시 미매칭 약 1.4%)은 같은 단지의 거래 기록(가격 제외 물리 속성)으로 채운다.
- `snapshot(tag)`는 `_학습기준` / `_최신` 열을 접미사 없는 이름(= `v2_trade` 열 이름)으로 꺼낸다.

In [2]:
def trade_profile(df):
    g = df.groupby("단지일련번호")
    out = g.agg(전용면적=("전용면적", "median"), 층=("층", "median"), 층_최대=("층", "max"), 거래수=("전용면적", "size"),
                건축년도_거래=("건축년도", "median"),
                역세권거리_m_v1=("역세권거리_m", "median"), 주요일자리거리_m_v1=("주요일자리거리_m", "median"),
                log평당가=("평당가", lambda x: np.log(x).median()))
    q = g["전용면적"].quantile([0.25, 0.75]).unstack()
    out["전용면적_IQR"] = q[0.75] - q[0.25]
    out["거래_소형비중"] = g["전용면적"].apply(lambda x: (x <= 60).mean())
    out["거래_대형비중"] = g["전용면적"].apply(lambda x: (x > 85).mean())
    out["기준연식_v1"] = (2025 - out["건축년도_거래"]).clip(lower=0)
    return out


train_profile = trade_profile(trades[trades["거래월일자"] < TRAIN_END])
new_profile = trade_profile(trades[trades["거래월일자"] >= TRAIN_END])
new_profile[["log평당가", "거래수"]] = np.nan  # 2026 거래는 가격 · 가중치 어디에도 쓰지 않는다
profile = pd.concat([train_profile, new_profile[~new_profile.index.isin(train_profile.index)]])
cx = cx.join(profile, on="단지일련번호", validate="one_to_one")
cx["기준연식"] = (2025 - cx["건축년도"]).clip(lower=0)  # V1과 같은 정의 (분석용 군집 · 유사도 Feature)
assert cx["전용면적"].notna().all() and cx.loc[cx["학습단지"], "log평당가"].notna().all()

FALLBACK = {"전용_중위": "전용면적", "소형60이하_비중": "거래_소형비중", "대형85초과_비중": "거래_대형비중",
            "평균층": "층_최대", "최고층": "층_최대"}
cx["주택특성_보정"] = cx[list(FALLBACK) + ["세대수"]].isna().any(axis=1)
cx["최고층_거래보정"] = cx["최고층"].isna()  # 건축물대장 층수가 없어 거래 최고층(실제 이하)으로 채운 단지
for col, src in FALLBACK.items():
    cx[col] = cx[col].fillna(cx[src])
cx["주요단지"] = cx["세대수"] >= MAIN_MIN_HOUSEHOLDS
print("05b 결측 보정 단지:", f"{cx['주택특성_보정'].sum():,}", f"({cx['주택특성_보정'].mean():.1%})",
      "| 세대수 결측", cx["세대수"].isna().sum(), "(규모 태그 없음, 주요 단지 아님으로 처리)")


def snapshot(tag):
    out = cx.copy()
    for col in [c for c in cx.columns if c.endswith(f"_{tag}")]:
        out[col.removesuffix(f"_{tag}")] = cx[col]
    return out.set_index("단지일련번호")


view_train = snapshot("학습기준")
view_latest = snapshot("최신")
train = view_train[view_train["학습단지"]]
main = train["주요단지"].to_numpy()
w_trade = train["거래수"].to_numpy(float)
y_price = train["log평당가"].to_numpy()
print("학습 단지:", f"{len(train):,}", "| 주요 단지(150세대 이상)", f"{main.sum():,} ({main.mean():.1%})",
      "| 주요 단지의 학습 구간 거래 비중", f"{w_trade[main].sum() / w_trade.sum():.1%}")
print("세대수 분포 (학습 단지):", {f"<{t}": f"{(train['세대수'] < t).mean():.1%}" for t in [50, 150, 300, 1000]},
      "| 1개 동", f"{(train['동수'] == 1).mean():.1%}")

05b 결측 보정 단지: 108 (1.4%) | 세대수 결측 89 (규모 태그 없음, 주요 단지 아님으로 처리)
학습 단지: 7,581 | 주요 단지(150세대 이상) 2,520 (33.2%) | 주요 단지의 학습 구간 거래 비중 85.2%
세대수 분포 (학습 단지): {'<50': '46.3%', '<150': '65.7%', '<300': '78.4%', '<1000': '93.5%'} | 1개 동 64.8%


## 2. 군집 진단: 왜 KMeans 유형을 사용자 태그로 쓰지 않는가

먼저 V1을 같은 함수로 재현(ARI 1.0)한 뒤, 주택특성 군집을 세 가지 학습 방식으로 비교한다.

- **현재:** 전체 단지를 같은 비중으로 학습 (첫 버전)
- **분리:** 주요 단지(150세대 이상)만 학습
- **가중:** 전체 단지를 세대수 가중으로 학습

모두 **주요 단지 안에서**(그 단지들의 라벨, 같은 표준화 공간) Silhouette · 가장 약한 군집 · 경계 단지 비율(Silhouette < 0) ·
최소 군집 비중 · 재학습 안정성(80% 부분표본 5회 ARI 최저)을 잰다.

In [3]:
LOG_COLS = {"전용면적", "전용면적_IQR", "전용_중위", "세대수", "역세권거리_m_v1", "주요일자리거리_m_v1",
            "역세권거리_m", "주요일자리거리_m", "최근접초_거리_m", "최근접학원가_거리_m", "반경1km_입시교과학원수"}


def raw_matrix(df, cols, fill):
    m = df[cols].astype(float).fillna(fill)
    for c in cols:
        if c in LOG_COLS:
            m[c] = np.log1p(m[c].clip(lower=0))
    return m.to_numpy()


def fit_axis(df, cols, k, weight=None, seed=SEED):
    fill = df[cols].median()
    scaler = StandardScaler().fit(raw_matrix(df, cols, fill))
    X = scaler.transform(raw_matrix(df, cols, fill))
    model = KMeans(k, n_init=20, random_state=seed).fit(X, sample_weight=weight)
    return {"cols": cols, "fill": fill, "scaler": scaler, "model": model, "k": k}


def axis_matrix(axis, df):
    return axis["scaler"].transform(raw_matrix(df, axis["cols"], axis["fill"]))


def predict_axis(axis, df):
    return axis["model"].predict(axis_matrix(axis, df))


v1_labels = pd.read_parquet(V1_MODEL_DIR / "complex_clusters.parquet").set_index("단지일련번호")
for col, cols in [("location_cluster", ["역세권거리_m_v1", "주요일자리거리_m_v1"]),
                  ("housing_cluster", ["전용면적", "전용면적_IQR", "기준연식_v1", "층"])]:
    ax = fit_axis(train, cols, 3)
    ari = adjusted_rand_score(v1_labels.loc[train.index, col], ax["model"].labels_)
    print(f"V1 재현 {col}: ARI {ari:.4f} | Silhouette {silhouette_score(axis_matrix(ax, train), ax['model'].labels_):.4f}")
    assert ari == 1.0

V1 재현 location_cluster: ARI 1.0000 | Silhouette 0.3725


V1 재현 housing_cluster: ARI 1.0000 | Silhouette 0.3320


In [4]:
HOUSING_SETS = {
    "면적 구성(소형·대형 비중)": ["전용_중위", "소형60이하_비중", "대형85초과_비중", "기준연식", "평균층", "세대수"],
    "최소(면적·연식·층·세대)": ["전용_중위", "기준연식", "평균층", "세대수"],
}
hh_weight = train["세대수"].fillna(train["세대수"].median()).to_numpy(float)


def fit_by_approach(df, cols, k, approach, seed=SEED):
    is_main = df["주요단지"].to_numpy()
    if approach == "분리":
        return fit_axis(df[is_main], cols, k, seed=seed)
    weight = df["세대수"].fillna(train["세대수"].median()).to_numpy(float) if approach == "가중" else None
    return fit_axis(df, cols, k, weight=weight, seed=seed)


def main_quality(labels_main, X_main, w_main):
    s = silhouette_samples(X_main, labels_main)
    per = pd.DataFrame({"l": labels_main, "s": s, "w": w_main}).groupby("l").agg(
        mean=("s", "mean"), neg=("s", lambda x: (x < 0).mean()), n=("s", "size"), w=("w", "sum"))
    return {"주요단지_Silhouette": s.mean(), "거래가중_Silhouette": np.average(s, weights=w_main),
            "최약군집_Silhouette": per["mean"].min(), "경계단지_최대비율": per["neg"].max(),
            "최소군집_거래비중": (per["w"] / per["w"].sum()).min()}


rng = np.random.default_rng(0)
rows = []
for set_name, cols in HOUSING_SETS.items():
    for approach in ["현재", "분리", "가중"]:
        for k in range(3, 7):
            ax = fit_by_approach(train, cols, k, approach)
            X_main = axis_matrix(ax, train[main])
            lab_main = ax["model"].predict(X_main)
            aris = []
            for b in range(5):
                idx = rng.choice(len(train), int(0.8 * len(train)), replace=False)
                sub = fit_by_approach(train.iloc[idx], cols, k, approach, seed=b)
                sub_main = train.iloc[idx]["주요단지"].to_numpy()
                Xs = axis_matrix(ax, train.iloc[idx][sub_main])
                aris.append(adjusted_rand_score(ax["model"].predict(Xs), sub["model"].predict(axis_matrix(sub, train.iloc[idx][sub_main]))))
            rows.append({"Feature": set_name, "방식": approach, "k": k, **main_quality(lab_main, X_main, w_trade[main]),
                         "재학습_ARI_최저": min(aris)})
housing_diag = pd.DataFrame(rows)
housing_diag.to_csv(MODEL_DIR / "housing_cluster_diagnosis.csv", index=False, encoding="utf-8-sig")
display(housing_diag.round(3))

,Feature,방식,k,주요단지_Silhouette,거래가중_Silhouette,최약군집_Silhouette,경계단지_최대비율,최소군집_거래비중,재학습_ARI_최저
0,면적 구성(소형·대형 비중),현재,3,0.151,0.174,0.129,0.225,0.004,0.972
1,면적 구성(소형·대형 비중),현재,4,0.106,0.128,0.052,0.387,0.007,0.962
2,면적 구성(소형·대형 비중),현재,5,0.093,0.122,0.014,0.452,0.006,0.956
3,면적 구성(소형·대형 비중),현재,6,0.117,0.100,0.053,0.399,0.004,0.958
4,면적 구성(소형·대형 비중),분리,3,0.273,0.252,0.165,0.174,0.095,0.904
5,면적 구성(소형·대형 비중),분리,4,0.251,0.235,0.182,0.130,0.093,0.952
6,면적 구성(소형·대형 비중),분리,5,0.263,0.249,0.193,0.149,0.024,0.959
7,면적 구성(소형·대형 비중),분리,6,0.267,0.262,0.201,0.097,0.024,0.886
8,면적 구성(소형·대형 비중),가중,3,0.221,0.213,0.136,0.218,0.185,0.870
9,면적 구성(소형·대형 비중),가중,4,0.270,0.227,0.095,0.290,0.085,0.872


In [5]:
# 다른 알고리즘: 주요 단지에서 밀도 기반(HDBSCAN) · 확률 기반(GMM)이 더 나은 구조를 찾는지
X_main = StandardScaler().fit_transform(raw_matrix(train[main], HOUSING_SETS["면적 구성(소형·대형 비중)"],
                                                   train[HOUSING_SETS["면적 구성(소형·대형 비중)"]].median()))
rows = []
for mcs in [25, 50, 100]:
    for ms in [5, 15]:
        for method in ["eom", "leaf"]:
            h = HDBSCAN(min_cluster_size=mcs, min_samples=ms, cluster_selection_method=method, copy=True).fit_predict(X_main)
            keep = h >= 0
            n_cl = len(set(h[keep]))
            rows.append({"알고리즘": f"HDBSCAN({mcs},{ms},{method})", "군집수": n_cl, "미배정_비율": 1 - keep.mean(),
                         "최대군집_비중": np.bincount(h[keep]).max() / len(h) if n_cl else np.nan,
                         "Silhouette(배정 단지)": silhouette_score(X_main[keep], h[keep]) if n_cl > 1 else np.nan})
for k in range(3, 7):
    gm = GaussianMixture(k, covariance_type="full", n_init=3, random_state=SEED).fit(X_main)
    lab = gm.predict(X_main)
    rows.append({"알고리즘": f"GMM(full, k={k})", "군집수": k, "미배정_비율": 0.0, "최대군집_비중": np.bincount(lab).max() / len(lab),
                 "Silhouette(배정 단지)": silhouette_score(X_main, lab)})
alt_diag = pd.DataFrame(rows)
display(alt_diag.round(3))

# Silhouette이 높은 HDBSCAN(eom)이 실제로 무엇을 나눴는지
h = HDBSCAN(min_cluster_size=25, min_samples=5, cluster_selection_method="eom", copy=True).fit_predict(X_main)
hcols = ["전용_중위", "세대수", "건축년도", "평균층"]
display(train[main].assign(h=h).groupby("h")[hcols].median().assign(단지수=pd.Series(h).value_counts().sort_index().values).round(1))

,알고리즘,군집수,미배정_비율,최대군집_비중,Silhouette(배정 단지)
0,"HDBSCAN(25,5,eom)",2,0.079,0.899,0.479
1,"HDBSCAN(25,5,leaf)",11,0.688,0.078,0.373
2,"HDBSCAN(25,15,eom)",2,0.127,0.855,0.505
3,"HDBSCAN(25,15,leaf)",7,0.813,0.055,0.504
4,"HDBSCAN(50,5,eom)",2,0.079,0.899,0.479
5,"HDBSCAN(50,5,leaf)",8,0.695,0.078,0.414
6,"HDBSCAN(50,15,eom)",2,0.338,0.631,0.441
7,"HDBSCAN(50,15,leaf)",3,0.860,0.055,0.478
8,"HDBSCAN(100,5,eom)",2,0.238,0.706,0.409
9,"HDBSCAN(100,5,leaf)",3,0.779,0.087,0.403


,전용_중위,세대수,건축년도,평균층,단지수
h,,,,,
-1,84.8,491.5,2005.5,21.8,200
0,84.8,406.0,2001.0,15.7,2265
1,15.6,216.0,2016.0,16.5,55


### 진단 결론

- **현재 방식(첫 버전)은 주요 단지를 구분하지 못한다.** 주요 단지 Silhouette 0.1 안팎, 경계 단지가 군집에 따라 40% 이상,
  가장 작은 군집은 주요 단지 거래의 1% 미만이다. 전체 단지 기준 0.35는 작은 단지들이 만든 점수였다.
- **분리 · 가중으로 바꿔도 0.25 안팎**이다. 분리는 평가 대상(주요 단지)에 맞춰 학습했으므로 유리한 비교인데도 이 수준이고,
  가중은 재학습마다 결과가 흔들리는 조합이 많다.
- **HDBSCAN은 Silhouette이 높게 나오는 설정에서도 주요 단지의 약 90%를 한 덩어리(전용 약 85㎡ · 약 400세대)로 묶고
  2% 남짓의 작은 무리(전용 15㎡대 신축 소형)만 떼어낸다.** 군집을 더 잘게 나누는 설정(leaf)은 단지 대부분을 미배정으로 남긴다.
  GMM은 Silhouette이 0 근처다.
- 즉 주요 단지의 주택 특성은 연속 분포다. 경계가 임의적인 군집을 사용자 태그로 내보내면 "왜 이 유형인가"를 설명할 수 없고,
  경계 단지는 재학습 · 데이터 갱신 때 유형이 바뀐다. → **사용자 태그는 규칙으로 만든다.**

## 3. 규칙 태그

| 축 | 태그 | 기준 | 근거 | 성격 |
|---|---|---|---|---|
| 규모 | 소규모 단지 | 150세대 미만 | 공동주택관리법 제2조제1항제2호: 300세대 이상 또는 150세대 이상(승강기 · 중앙집중식 난방 · 주상복합)만 의무관리 대상 → 150세대 미만은 의무관리 대상이 될 수 없음 (입주자 동의 지정 제외) | 법령 |
| 규모 | 대단지 | 1,000세대 이상 | 법적 정의 없음 | 관행 |
| 연식 | 신축 / 준신축 | 준공 후 5년 / 10년 이내 | 법적 정의 없음 | 관행 |
| 연식 | 재건축 연한 도달 | 준공연도 · 층수별 경과연수 (5층 이상: 1986년 이후 준공 30년, 4층 이하: 1991년 이후 준공 30년, 그 전은 20~29년) | 서울특별시 도시 및 주거환경정비 조례 [별표 1] 공동주택 노후 · 불량건축물 기준 | 조례 |
| 재건축 | 재건축 추진 중 | 정비구역 지정 이후 단계 (구역지정 · 추진위 · 조합설립 · 사업시행 · 관리처분 · 이주 · 착공) | 서울시 정비사업 추진현황 (05b, 단계별 고시일) | 공식 자료 |
| 평형 | 소형 / 중형 / 대형 위주 | 단지 전체 호 전용면적 중위 60㎡ 이하 / 60~85㎡ / 85㎡ 초과 | 85㎡: 주택법 제2조제6호 국민주택규모. 60㎡: HUG · 부동산R114 등 공통 구분 | 법령 + 관행 |
| 교통 | 역 출입구 250m 이내 / 500m 이내 | 가장 가까운 역 **출입구** 직선거리 (06 `역출입구거리_m`, 서울교통빅데이터플랫폼 출입구 좌표) | 거리값은 서울시 역세권 장기전세주택 운영기준(1차 250m · 2차 500m)에서 가져왔다. 공식 판정은 승강장 경계 ↔ 사업대상지 경계로 재고 사업지마다 심의로 정하며, 승강장 경계 데이터는 공개되지 않는다 → **공식 역세권 판정이 아니므로 "역세권" 이름을 쓰지 않는다** | 측정값 |
| 교육 | 초등학교 가까움 | 최근접 초등학교 300m 이내 (직선, 도보 약 5분) | 법정 통학거리(도시 · 군계획시설 규칙: 1.5km 이내)는 서울 단지 대부분이 해당해 구분이 안 됨 | 관행 |
| 교육 | 학원가 인접 | 02c 학원가(학원 밀집 상위 행정동 11곳) 1km 이내 | 02c 정의 | 데이터 기준 |

- 관행 · 데이터 기준 태그는 웹에서 기준값을 함께 보여준다 (예: "대단지 · 1,240세대"). 기준값은 `tag_rules.json` 한 곳에서 관리한다.
- 직주근접은 태그로 만들지 않는다. 사용자마다 직장이 달라 05b 결정대로 웹에서 실시간 통근시간으로 제공한다.

In [6]:
TAG_RULES = {
    "소규모 단지": {"축": "규모", "기준": "세대수 < 150", "근거": "공동주택관리법 제2조제1항제2호 (의무관리 대상 최소 규모)", "성격": "법령"},
    "대단지": {"축": "규모", "기준": "세대수 >= 1000", "근거": "관행 (법적 정의 없음)", "성격": "관행"},
    "신축": {"축": "연식", "기준": "준공 후 5년 이내", "근거": "관행", "성격": "관행"},
    "준신축": {"축": "연식", "기준": "준공 후 6~10년", "근거": "관행", "성격": "관행"},
    "재건축 연한 도달": {"축": "연식", "기준": "서울 정비조례 별표1 노후·불량 기준 연수 이상 경과",
                    "근거": "서울특별시 도시 및 주거환경정비 조례 [별표 1]", "성격": "조례"},
    "재건축 추진 중": {"축": "재건축", "기준": "정비구역 지정 이후 단계", "근거": "서울시 정비사업 추진현황 (05b)", "성격": "공식 자료"},
    "소형 위주": {"축": "평형", "기준": "전용 중위 <= 60㎡", "근거": "업계 공통 구분 (HUG · 부동산R114)", "성격": "관행"},
    "중형 위주": {"축": "평형", "기준": "60㎡ < 전용 중위 <= 85㎡", "근거": "주택법 제2조제6호 국민주택규모 85㎡", "성격": "법령"},
    "대형 위주": {"축": "평형", "기준": "전용 중위 > 85㎡", "근거": "주택법 제2조제6호 국민주택규모 초과", "성격": "법령"},
    "역 출입구 250m 이내": {"축": "교통", "기준": "가장 가까운 역 출입구 <= 250m (직선)",
                       "근거": "거리값은 서울시 역세권 운영기준 1차 250m. 공식 판정(승강장 경계 ↔ 부지 경계, 심의)은 아님", "성격": "측정값"},
    "역 출입구 500m 이내": {"축": "교통", "기준": "250m < 가장 가까운 역 출입구 <= 500m (직선)",
                       "근거": "거리값은 서울시 역세권 운영기준 2차 500m. 공식 판정(승강장 경계 ↔ 부지 경계, 심의)은 아님", "성격": "측정값"},
    "초등학교 가까움": {"축": "교육", "기준": "최근접 초등학교 <= 300m (직선)", "근거": "관행 (도보 약 5분)", "성격": "관행"},
    "학원가 인접": {"축": "교육", "기준": "02c 학원가 <= 1km", "근거": "02c 학원가 정의", "성격": "데이터 기준"},
}
for rule in TAG_RULES.values():
    rule["웹_기준값_조정"] = rule["성격"] in ("관행", "데이터 기준")  # 법적 정의가 없는 기준은 웹 필터에서 사용자가 바꿀 수 있게 한다
REBUILD_STAGE_NAMES = {1: "구역지정", 2: "추진위원회", 3: "조합설립", 4: "사업시행인가", 5: "관리처분인가", 6: "이주", 7: "착공"}


def rebuild_age_required(built_year, floors):
    # 서울 정비조례 별표1: 철근콘크리트 등 공동주택 노후·불량건축물 기준 연수
    if built_year <= 1981:
        return 20
    if floors >= 5:
        return {1982: 22, 1983: 24, 1984: 26, 1985: 28}.get(int(built_year), 30)
    return {1982: 21, 1983: 22, 1984: 23, 1985: 24, 1986: 25, 1987: 26, 1988: 27, 1989: 28, 1990: 29}.get(int(built_year), 30)


def rule_tags(view, year):
    age = year - view["건축년도"]
    floors = view["최고층"].fillna(5)
    need = [rebuild_age_required(b, f) for b, f in zip(view["건축년도"], floors)]
    households = view["세대수"]
    tags = pd.DataFrame(index=view.index)
    tags["규모"] = np.select([households < 150, households >= 1000], ["소규모 단지", "대단지"], default="")
    tags.loc[households.isna(), "규모"] = ""
    tags["연식"] = np.select([age <= 5, age <= 10, age >= np.array(need)], ["신축", "준신축", "재건축 연한 도달"], default="")
    tags["재건축"] = np.where(view["재건축_단계"] >= 1, "재건축 추진 중", "")
    tags["평형"] = np.select([view["전용_중위"] <= 60, view["전용_중위"] <= 85], ["소형 위주", "중형 위주"], default="대형 위주")
    tags["교통"] = np.select([view["역출입구거리_m"] <= 250, view["역출입구거리_m"] <= 500], ["역 출입구 250m 이내", "역 출입구 500m 이내"], default="")
    tags["교육_초등"] = np.where(view["최근접초_거리_m"] <= 300, "초등학교 가까움", "")
    tags["교육_학원가"] = np.where(view["최근접학원가_거리_m"] <= 1000, "학원가 인접", "")
    tags["재건축_단계명"] = view["재건축_단계"].map(REBUILD_STAGE_NAMES).fillna("")
    tags["태그"] = tags[["규모", "연식", "재건축", "평형", "교통", "교육_초등", "교육_학원가"]].apply(
        lambda r: " · ".join(t for t in r if t), axis=1)
    return tags


TAG_COLS = ["규모", "연식", "재건축", "평형", "교통", "교육_초등", "교육_학원가"]
tags_train = rule_tags(view_train, SNAP_YEAR["학습기준"])
tags_latest = rule_tags(view_latest, SNAP_YEAR["최신"])
assert set(tags_latest[TAG_COLS].stack().unique()) - {""} <= set(TAG_RULES)

coverage = pd.DataFrame({
    "전체 학습 단지": tags_train.loc[train.index, TAG_COLS].apply(lambda s: s.value_counts(normalize=True)).stack(),
    "주요 단지": tags_train.loc[train.index[main], TAG_COLS].apply(lambda s: s.value_counts(normalize=True)).stack(),
}).drop(index="", level=0, errors="ignore")
coverage.index = coverage.index.swaplevel()
display((coverage.sort_index() * 100).round(1).rename(columns=lambda c: c + " (%)"))

전체 학습 단지 (%)  주요 단지 (%)
교육_초등  초등학교 가까움               39.4       46.4
교육_학원가 학원가 인접                  7.0       10.2
교통     역 출입구 250m 이내          27.0       25.0
       역 출입구 500m 이내          35.8       35.7
규모     대단지                     5.4       16.2
       소규모 단지                 65.7        NaN
연식     신축                      6.1        7.1
       재건축 연한 도달              14.6       25.9
       준신축                     6.8        7.8
재건축    재건축 추진 중                1.4        3.9
평형     대형 위주                  18.7       14.4
       소형 위주                  25.6       23.3
       중형 위주                  55.8       62.3

In [7]:
# 재건축 연한 판정 점검: 별표1 경계 사례
for built, floors, year, expect in [(1981, 5, 2001, True), (1985, 12, 2012, False), (1985, 12, 2013, True),
                                    (1986, 15, 2015, False), (1986, 15, 2016, True), (1990, 4, 2019, True), (1991, 3, 2020, False)]:
    assert (year - built >= rebuild_age_required(built, floors)) == expect, (built, floors, year)
print("재건축 연한 경계 사례 7건 통과")
# 층수에 따라 기준이 갈리는 것은 1982~1990년 준공 단지뿐이다. 그중 층수를 거래 최고층으로 채운 단지를 따로 센다
affected = cx["최고층_거래보정"] & cx["건축년도"].between(1982, 1990)
print(f"층수를 거래 최고층으로 채운 단지 {cx['최고층_거래보정'].sum()}곳 중 판정에 영향이 있을 수 있는 1982~1990년 준공 {affected.sum()}곳",
      "| 그중 거래 최고층 4층 이하(4층 이하 기준 적용)", int((affected & (cx["최고층"] <= 4)).sum()))

examples = ["은마", "헬리오시티", "목동신시가지7", "래미안대치팰리스", "파크리오"]
ex = tags_latest.join(view_latest[["단지명", "시군구명", "세대수", "건축년도", "전용_중위", "역출입구거리_m", "최근접초_거리_m",
                                    "최근접학원가_거리_m", "재건축_단계"]])
display(ex[ex["단지명"].isin(examples)][["단지명", "시군구명", "태그", "재건축_단계명", "세대수", "건축년도", "전용_중위",
                                       "역출입구거리_m", "최근접초_거리_m", "최근접학원가_거리_m"]].round(0))

재건축 연한 경계 사례 7건 통과
층수를 거래 최고층으로 채운 단지 92곳 중 판정에 영향이 있을 수 있는 1982~1990년 준공 13곳 | 그중 거래 최고층 4층 이하(4층 이하 기준 적용) 3


,단지명,시군구명,태그,재건축_단계명,세대수,건축년도,전용_중위,역출입구거리_m,최근접초_거리_m,최근접학원가_거리_m
단지일련번호,,,,,,,,,,
11470-23,목동신시가지7,양천구,대단지 · 재건축 연한 도달 · 재건축 추진 중 · 중형 위주 · 역 출입구 500...,추진위원회,2550.0,1986,67.0,346.0,501.0,1288.0
11680-218,은마,강남구,대단지 · 재건축 연한 도달 · 재건축 추진 중 · 중형 위주 · 역 출입구 250...,조합설립,4424.0,1979,77.0,229.0,302.0,547.0
11680-4394,래미안대치팰리스,강남구,대단지 · 대형 위주 · 역 출입구 500m 이내 · 학원가 인접,,1608.0,2015,94.0,311.0,450.0,304.0
11710-6266,파크리오,송파구,대단지 · 중형 위주 · 역 출입구 250m 이내 · 초등학교 가까움,,6864.0,2008,85.0,192.0,258.0,2023.0
11710-8865,헬리오시티,송파구,대단지 · 준신축 · 중형 위주 · 역 출입구 500m 이내 · 초등학교 가까움,,9510.0,2018,85.0,496.0,293.0,2160.0


## 4. 규칙 태그 vs KMeans 유형 (A안 vs B안)

같은 기준으로 비교한다. 평가는 주요 단지 기준이다.

1. **재학습 안정성:** 규칙은 학습이 없어 항상 같다. KMeans는 부분표본 재학습 ARI.
2. **데이터 갱신(학습기준 → 최신)에 따른 변경:** 규칙은 실제 변화(역 개통 · 연식 경과 · 학원 수)만 반영한다.
3. **경계 근처 단지:** 규칙은 기준값 ±10% 안의 단지, KMeans는 Silhouette < 0인 단지.
   규칙은 경계 근처라도 "1,050세대라 대단지"처럼 이유가 분명하고, KMeans 경계 단지는 이유를 설명할 수 없다.
4. **사후 가격 설명력:** 주요 단지 평당가(log)의 그룹 간 분산 비율. 그룹 수가 다르므로 그룹 수를 보정한 값(adjusted)을 함께 본다.

In [8]:
def eta2(y, groups, weights=None, adjusted=False):
    d = pd.DataFrame({"y": y, "g": groups, "w": np.ones(len(y)) if weights is None else weights})
    mean = np.average(d["y"], weights=d["w"])
    gm = d.groupby("g").apply(lambda x: np.average(x["y"], weights=x["w"]), include_groups=False)
    between = (d["w"] * (d["g"].map(gm) - mean) ** 2).sum()
    total = (d["w"] * (d["y"] - mean) ** 2).sum()
    e = between / total
    if adjusted:
        n, k = len(d), d["g"].nunique()
        e = 1 - (1 - e) * (n - 1) / (n - k)
    return e


def near_threshold(values, thresholds, tol=0.1):
    v = np.asarray(values, float)
    return np.any([np.abs(v - t) <= tol * t for t in thresholds], axis=0)


tm = tags_train.loc[train.index[main]]
vm = train[main]
age_m = SNAP_YEAR["학습기준"] - vm["건축년도"]
rule_groups = {
    "규모": (tm["규모"], near_threshold(vm["세대수"], [1000])),
    "연식": (tm["연식"], near_threshold(age_m, [5, 10]) | (np.abs(age_m - 30) <= 2)),
    "평형": (tm["평형"], near_threshold(vm["전용_중위"], [60, 85])),
    "교통": (tm["교통"], near_threshold(vm["역출입구거리_m"], [250, 500])),
    "교육": (tm["교육_초등"] + "|" + tm["교육_학원가"], near_threshold(vm["최근접초_거리_m"], [300]) | near_threshold(vm["최근접학원가_거리_m"], [1000])),
}
housing_rule_combo = tm["규모"] + "|" + tm["연식"] + "|" + tm["평형"]
cmp_rows = []
for name, (groups, near) in rule_groups.items():
    cmp_rows.append({"안": "A 규칙", "축": name, "그룹수": groups.nunique(), "재학습 안정성": 1.0, "경계 근처 비율": near.mean(),
                     "가격 eta2(거래가중)": eta2(y_price[main], groups.to_numpy(), w_trade[main]),
                     "가격 eta2(adjusted)": eta2(y_price[main], groups.to_numpy(), adjusted=True)})
cmp_rows.append({"안": "A 규칙", "축": "주택(규모×연식×평형)", "그룹수": housing_rule_combo.nunique(), "재학습 안정성": 1.0,
                 "경계 근처 비율": (rule_groups["규모"][1] | rule_groups["연식"][1] | rule_groups["평형"][1]).mean(),
                 "가격 eta2(거래가중)": eta2(y_price[main], housing_rule_combo.to_numpy(), w_trade[main]),
                 "가격 eta2(adjusted)": eta2(y_price[main], housing_rule_combo.to_numpy(), adjusted=True)})

best_b = {"주택(KMeans 현재, k=5)": ("현재", 5), "주택(KMeans 분리, k=4)": ("분리", 4)}
for name, (approach, k) in best_b.items():
    ax = fit_by_approach(train, HOUSING_SETS["면적 구성(소형·대형 비중)"], k, approach)
    lab = ax["model"].predict(axis_matrix(ax, train[main]))
    row = housing_diag[(housing_diag["Feature"] == "면적 구성(소형·대형 비중)") & (housing_diag["방식"] == approach) & (housing_diag["k"] == k)].iloc[0]
    s = silhouette_samples(axis_matrix(ax, train[main]), lab)
    cmp_rows.append({"안": "B KMeans", "축": name, "그룹수": k, "재학습 안정성": row["재학습_ARI_최저"], "경계 근처 비율": (s < 0).mean(),
                     "가격 eta2(거래가중)": eta2(y_price[main], lab, w_trade[main]), "가격 eta2(adjusted)": eta2(y_price[main], lab, adjusted=True)})
comparison = pd.DataFrame(cmp_rows)
comparison.to_csv(MODEL_DIR / "rule_vs_kmeans_comparison.csv", index=False, encoding="utf-8-sig")
display(comparison.round(3))

churn = pd.Series({c: (tags_train.loc[train.index, c] != tags_latest.loc[train.index, c]).mean() for c in TAG_COLS}, name="학습기준→최신 변경 비율")
display((churn * 100).round(2).to_frame("변경 비율 (%)"))

# 평형 경계 근처 비율이 높은 이유: 국민평형 등은 법정 기준선 바로 아래로 설계된다 → 기준선 양쪽의 애매한 단지는 드물다
area = vm["전용_중위"]
print("주요 단지 전용 중위 — 84~85㎡", f"{area.between(84, 85).mean():.1%}", "| 85㎡ 초과~87㎡", f"{((area > 85) & (area <= 87)).mean():.1%}",
      "| 59~60㎡", f"{area.between(59, 60).mean():.1%}", "| 60㎡ 초과~62㎡", f"{((area > 60) & (area <= 62)).mean():.1%}")

,안,축,그룹수,재학습 안정성,경계 근처 비율,가격 eta2(거래가중),가격 eta2(adjusted)
0,A 규칙,규모,2,1.000,0.051,0.035,0.028
1,A 규칙,연식,4,1.000,0.208,0.088,0.085
2,A 규칙,평형,3,1.000,0.705,0.053,0.059
3,A 규칙,교통,3,1.000,0.188,0.062,0.059
4,A 규칙,교육,4,1.000,0.170,0.057,0.044
5,A 규칙,주택(규모×연식×평형),24,1.000,0.763,0.197,0.186
6,B KMeans,"주택(KMeans 현재, k=5)",5,0.956,0.360,0.038,0.035
7,B KMeans,"주택(KMeans 분리, k=4)",4,0.952,0.065,0.125,0.137


,변경 비율 (%)
규모,0.00
연식,5.67
재건축,0.40
평형,0.00
교통,0.00
교육_초등,0.00
교육_학원가,0.00


주요 단지 전용 중위 — 84~85㎡ 49.6% | 85㎡ 초과~87㎡ 0.2% | 59~60㎡ 10.9% | 60㎡ 초과~62㎡ 0.4%


### 비교 결론

- **안정성:** 규칙 태그는 재학습과 무관하다. 학습기준 → 최신 사이 변경은 연식 경과 · 역 개통 · 학원 수처럼 실제로 바뀐 것만 반영한다.
- **경계:** 기준값 ±10% 근처 비율은 규칙이 더 높게 나온다. 대부분 평형 때문인데, 주요 단지의 약 절반이 전용 중위 84~85㎡로
  법정 기준선(85㎡) **바로 아래**에 몰려 있고 기준선 바로 위(85~87㎡)는 0.2%뿐이다. 국민평형을 기준선 아래로 설계하기 때문이라
  판정이 애매한 단지는 거의 없다. 나머지 태그도 기준값과 실제 값을 함께 보여주므로 사용자가 직접 판단할 수 있다.
  KMeans 경계 단지(분리 방식 약 7%, 현재 방식 약 36%)는 왜 그 유형인지 설명할 수 없다.
- **가격 설명력:** 주택 규칙 조합(규모 × 연식 × 평형)이 분리 KMeans와 비슷하거나 그 이상이다. 그룹 수 보정 후에도 같은 결론이다.
  가격을 쓰지 않은 해석 가능한 기준만으로 같은 정도의 시장 구분을 얻는다.
- 따라서 **웹 태그는 규칙(A안)**으로 확정한다.

## 5. 분석용 KMeans (입지 · 교육 축)

입지 · 교육 축은 위치에 따른 연속값이라 법적 경계가 없는 부분(예: 업무지구 접근성과 역 접근성의 조합)을 요약하는 데 쓴다.
웹 태그가 아니라 **시장 구조 분석 · 11 회귀 Ablation · 11 오차 분석**용이다.

- 주택특성 KMeans는 2장 진단대로 주요 단지에서 구분력이 없어 두지 않는다. 11에서 주택 특성은 원본 Feature와 규칙 태그로 본다.
- 입지 · 교육 축은 학습 방식(현재 · 분리 · 가중)에 따른 차이가 작아(아래 표) 전체 단지로 학습한다.
  위치 기반 축이라 작은 단지도 같은 동네의 정상적인 일원이다.
- K는 첫 버전의 비교와 같다: 입지 = V1 구성(역세권 + 주요일자리) K=3, 교육 = 초등학교 거리 · 반경 1km 입시교과 학원 수 · 학원가 거리 K=5.

In [9]:
AXES = {
    "location": {"label": "입지", "cols": ["역세권거리_m", "주요일자리거리_m"], "k": 3},
    "education": {"label": "교육", "cols": ["최근접초_거리_m", "반경1km_입시교과학원수", "최근접학원가_거리_m"], "k": 5},
}
rows = []
for key, spec in AXES.items():
    for approach in ["현재", "분리", "가중"]:
        ax = fit_by_approach(train, spec["cols"], spec["k"], approach)
        X_m = axis_matrix(ax, train[main])
        rows.append({"축": spec["label"], "방식": approach, **main_quality(ax["model"].predict(X_m), X_m, w_trade[main])})
display(pd.DataFrame(rows).round(3))

axes = {key: fit_axis(train, spec["cols"], spec["k"]) for key, spec in AXES.items()}
labels_train = {key: axes[key]["model"].labels_ for key in AXES}


def profile_of(key):
    p = train.assign(c=labels_train[key]).groupby("c")[AXES[key]["cols"]].median()
    p["단지수"] = np.bincount(labels_train[key])
    p["비중_%"] = p["단지수"] / len(train) * 100
    return p


def name_location(p):
    names = {p["주요일자리거리_m"].idxmin(): "직주근접 도심형"}
    rest = p.drop(index=list(names))
    names[rest["역세권거리_m"].idxmin()] = "역 인접 생활주거형"
    names.update({c: "일반 주거형" for c in p.index if c not in names})
    return names


def name_education(p):
    names = {p["반경1km_입시교과학원수"].idxmax(): "학원가 핵심형"}
    for col, label, how in [("최근접초_거리_m", "초등학교 인접형", "min"), ("최근접학원가_거리_m", "학원가 인접형", "min"),
                            ("반경1km_입시교과학원수", "동네 학원 밀집형", "max")]:
        rest = p.drop(index=list(names))
        names[getattr(rest[col], f"idx{how}")()] = label
    names.update({c: "교육시설 보통형" for c in p.index if c not in names})
    return names


profiles = {}
for key, namer in [("location", name_location), ("education", name_education)]:
    p = profile_of(key)
    axes[key]["names"] = namer(p)
    assert len(set(axes[key]["names"].values())) == AXES[key]["k"]
    X = axis_matrix(axes[key], train)
    s = silhouette_samples(X, labels_train[key])
    per = pd.DataFrame({"c": labels_train[key], "s": s}).groupby("c")["s"].agg(평균_Silhouette="mean", 경계단지_비율=lambda x: (x < 0).mean())
    p = p.join(per)
    p.insert(0, "유형", p.index.map(axes[key]["names"]))
    profiles[key] = p
    print(f"\n[{AXES[key]['label']}] 전체 Silhouette {s.mean():.3f} | Davies-Bouldin {davies_bouldin_score(X, labels_train[key]):.3f}")
    display(p.round(3))

,축,방식,주요단지_Silhouette,거래가중_Silhouette,최약군집_Silhouette,경계단지_최대비율,최소군집_거래비중
0,입지,현재,0.348,0.344,0.310,0.040,0.220
1,입지,분리,0.353,0.349,0.293,0.021,0.223
2,입지,가중,0.344,0.340,0.292,0.058,0.244
3,교육,현재,0.268,0.262,0.197,0.164,0.144
4,교육,분리,0.271,0.271,0.225,0.121,0.131
5,교육,가중,0.269,0.275,0.218,0.129,0.137



[입지] 전체 Silhouette 0.360 | Davies-Bouldin 0.942


,유형,역세권거리_m,주요일자리거리_m,단지수,비중_%,평균_Silhouette,경계단지_비율
c,,,,,,,
0,일반 주거형,664.539,5342.398,3625,47.817,0.413,0.000
1,직주근접 도심형,396.453,1566.865,1818,23.981,0.296,0.064
2,역 인접 생활주거형,265.053,4944.935,2138,28.202,0.325,0.016



[교육] 전체 Silhouette 0.286 | Davies-Bouldin 1.046


,유형,최근접초_거리_m,반경1km_입시교과학원수,최근접학원가_거리_m,단지수,비중_%,평균_Silhouette,경계단지_비율
c,,,,,,,,
0,교육시설 보통형,438.800,47.0,3814.282,1567,20.670,0.254,0.056
1,동네 학원 밀집형,358.475,118.0,5387.279,1812,23.902,0.298,0.004
2,학원가 인접형,422.603,120.0,1979.439,2013,26.553,0.338,0.000
3,초등학교 인접형,167.925,86.0,3312.259,1595,21.039,0.241,0.102
4,학원가 핵심형,283.081,314.5,719.614,594,7.835,0.277,0.096


In [10]:
for key in AXES:
    for tag, view in [("학습기준", view_train), ("최신", view_latest)]:
        lab = predict_axis(axes[key], view)
        cx[f"{key}_cluster_{tag}"] = lab
        cx[f"{key}_유형_{tag}"] = pd.Series(lab).map(axes[key]["names"]).to_numpy()
    assert (cx.loc[cx["학습단지"], f"{key}_cluster_학습기준"].to_numpy() == labels_train[key]).all()
    print(f"{AXES[key]['label']}: 학습기준 → 최신 유형 변경 {(cx[f'{key}_cluster_학습기준'] != cx[f'{key}_cluster_최신']).mean():.2%}")

# 11 회귀 Ablation용: 거래 시점 열로 배정 (역 = 계약 시점 운영 역, 학원 = 계약 시점 사용 가능 스냅샷)
trade_cols = ["transaction_id", "단지일련번호", "거래월일자"] + sorted({c for k in AXES for c in AXES[k]["cols"]})
v2_trade = pd.read_parquet(PROCESSED_DIR / "v2_trade.parquet", columns=trade_cols)
trade_clusters = v2_trade[["transaction_id"]].copy()
for key in AXES:
    trade_clusters[f"{key}_cluster"] = predict_axis(axes[key], v2_trade)
assert len(trade_clusters) == len(v2_trade) and trade_clusters.notna().all().all()

static = v2_trade[["transaction_id", "단지일련번호", "거래월일자", "역세권거리_m", "주요일자리거리_m"]].merge(trade_clusters, on="transaction_id").merge(
    cx[["단지일련번호", "location_cluster_학습기준", "education_cluster_학습기준", "역세권거리_m_학습기준", "주요일자리거리_m_학습기준"]],
    on="단지일련번호")
loc_diff = static[static["location_cluster"] != static["location_cluster_학습기준"]]
# 차이의 원인: 역 개통 전 거래(거래 시점 역 거리가 더 멂) 또는 그 시점 주요 일자리 분기 목록이 2025-07 기준과 다름 (06)
cause_station = loc_diff["역세권거리_m"] > loc_diff["역세권거리_m_학습기준"] + 1e-6
cause_job = (loc_diff["주요일자리거리_m"] - loc_diff["주요일자리거리_m_학습기준"]).abs() > 1e-6
assert (cause_station | cause_job).all()
print(f"입지 라벨이 단지 라벨(2025-07)과 다른 거래 {len(loc_diff):,}건 = 역 개통 전 {cause_station.sum():,}건 · "
      f"주요 일자리 목록 시점 차이 {(cause_job & ~cause_station).sum():,}건, 계약연도",
      pd.to_datetime(loc_diff["거래월일자"]).dt.year.value_counts().sort_index().to_dict())
print(f"교육 라벨이 단지 라벨과 다른 거래 {(static['education_cluster'] != static['education_cluster_학습기준']).mean():.2%} (학원 스냅샷 시점 차이)")

입지: 학습기준 → 최신 유형 변경 0.00%
교육: 학습기준 → 최신 유형 변경 4.78%
입지 라벨이 단지 라벨(2025-07)과 다른 거래 4,386건 = 역 개통 전 215건 · 주요 일자리 목록 시점 차이 4,171건, 계약연도 {2021: 669, 2022: 457, 2023: 513, 2024: 2177, 2025: 570}
교육 라벨이 단지 라벨과 다른 거래 3.27% (학원 스냅샷 시점 차이)


## 6. 유사 단지 (KNN)

목록마다 목적을 나누고, 목적에 맞는 기준으로 검증한다. 좌표와 가격은 유사도에 넣지 않는다.

| 목록 | 목적 | 검증 |
|---|---|---|
| 같은 구에서 비슷한 단지 (기본) | 실거주 비교 | 이웃 평당가 차이 (같은 시장 안에서 비슷한 단지는 가격대도 비슷해야 함) |
| 서울 전체에서 조건이 비슷한 단지 | 다른 지역 대안 탐색 (가격은 달라도 됨) | 이웃과 규칙 태그(규모 · 연식 · 평형 · 교통 · 교육)가 같은 비율 |

거리 공간: 입지 · 주택 · 교육 Feature를 축마다 표준화하고 `√(축 Feature 수)`로 나눠 세 축이 같은 비중을 갖게 한다.

In [11]:
K_NEIGHBORS = 5
SIM_AXES = {"입지": AXES["location"]["cols"], "주택": HOUSING_SETS["면적 구성(소형·대형 비중)"], "교육": AXES["education"]["cols"]}
sim_scalers = {name: fit_axis(train, cols, 2) for name, cols in SIM_AXES.items()}  # 표준화 · 결측값만 사용 (KMeans는 쓰지 않음)


def sim_space(df):
    return np.hstack([axis_matrix(sim_scalers[n], df) / np.sqrt(len(c)) for n, c in SIM_AXES.items()])


def knn(Z, groups=None):
    # Feature가 완전히 같은 단지가 있으면 자기 자신이 첫 번째가 아닐 수 있어, 위치와 관계없이 자기 자신을 뺀다
    out = np.zeros((len(Z), K_NEIGHBORS), int)
    for ix in ([np.arange(len(Z))] if groups is None else [np.where(groups == g)[0] for g in np.unique(groups)]):
        _, idx = NearestNeighbors(n_neighbors=K_NEIGHBORS + 1).fit(Z[ix]).kneighbors(Z[ix])
        for a, row in enumerate(idx):
            out[ix[a]] = ix[[j for j in row if j != a][:K_NEIGHBORS]]
    return out


def price_gap_pct(idx, rows):
    gap = np.abs(y_price[idx[rows]] - y_price[rows][:, None]).mean(axis=1)
    return (np.exp(np.median(gap)) - 1) * 100


def tag_agreement(idx, rows, cols=("규모", "연식", "평형", "교통", "교육_초등", "교육_학원가")):
    t = tags_train.loc[train.index, list(cols)].to_numpy()
    return (t[idx[rows]] == t[rows][:, None, :]).mean()


Z = sim_space(train)
gu = train["시군구명"].to_numpy()
rows_main = np.where(main)[0]
idx_seoul, idx_gu = knn(Z), knn(Z, gu)
assert (idx_seoul != np.arange(len(Z))[:, None]).all() and (idx_gu != np.arange(len(Z))[:, None]).all()
coords = np.radians(train[["위도", "경도"]].to_numpy())
_, geo = BallTree(coords, metric="haversine").query(coords, k=K_NEIGHBORS + 1)
rand = np.array([rng.choice(np.delete(np.arange(len(train)), i), K_NEIGHBORS, replace=False) for i in range(len(train))])
rand_gu = np.array([rng.choice(np.setdiff1d(np.where(gu == gu[i])[0], [i]), K_NEIGHBORS, replace=False) for i in range(len(train))])
knn_eval = pd.DataFrame({
    "평당가 차이 % (중앙, 주요 단지)": {n: price_gap_pct(i, rows_main) for n, i in
                              [("무작위", rand), ("같은 구 무작위", rand_gu), ("좌표 최근접", geo[:, 1:]), ("KNN 서울 전체", idx_seoul), ("KNN 같은 구", idx_gu)]},
    "규칙 태그 일치율 (주요 단지)": {n: tag_agreement(i, rows_main) for n, i in
                           [("무작위", rand), ("같은 구 무작위", rand_gu), ("좌표 최근접", geo[:, 1:]), ("KNN 서울 전체", idx_seoul), ("KNN 같은 구", idx_gu)]},
})
knn_eval.to_csv(MODEL_DIR / "knn_evaluation.csv", encoding="utf-8-sig")
display(knn_eval.round(3))
assert knn_eval.loc["KNN 같은 구", "평당가 차이 % (중앙, 주요 단지)"] < knn_eval.loc["좌표 최근접", "평당가 차이 % (중앙, 주요 단지)"]
assert knn_eval.loc["KNN 서울 전체", "규칙 태그 일치율 (주요 단지)"] > knn_eval.loc["좌표 최근접", "규칙 태그 일치율 (주요 단지)"]

,"평당가 차이 % (중앙, 주요 단지)",규칙 태그 일치율 (주요 단지)
무작위,65.105,0.477
같은 구 무작위,45.927,0.494
좌표 최근접,26.657,0.650
KNN 서울 전체,23.544,0.804
KNN 같은 구,18.116,0.764


- **같은 구 KNN**은 주요 단지에서 평당가 차이가 가장 작다. 좌표상 가장 가까운 단지보다도 비슷한 가격대의 단지를 찾는다 → 기본 목록.
- **서울 전체 KNN**은 가격 차이는 크지만(다른 지역이라 당연함) 규칙 태그 일치율이 가장 높다. "조건은 비슷한 다른 지역 대안"이라는 목적에 맞다.
  웹에서는 가격이 다르다는 점을 함께 보여준다 (예: "조건이 비슷하고 평당가가 낮은 단지").

In [12]:
latest = view_latest.loc[cx["단지일련번호"]]
Z_all = sim_space(latest)
ids = latest.index.to_numpy()
records = []
for scope, groups in [("서울 전체", None), ("같은 구", latest["시군구명"].to_numpy())]:
    idx = knn(Z_all, groups)
    dist = np.linalg.norm(Z_all[idx] - Z_all[:, None, :], axis=2)
    for r in range(K_NEIGHBORS):
        records.append(pd.DataFrame({"단지일련번호": ids, "범위": scope, "순위": r + 1, "유사단지일련번호": ids[idx[:, r]], "거리": dist[:, r]}))
similar = pd.concat(records, ignore_index=True)
assert similar.groupby(["단지일련번호", "범위"]).size().eq(K_NEIGHBORS).all()
assert (similar["단지일련번호"] != similar["유사단지일련번호"]).all()

name_of = cx.set_index("단지일련번호")["단지명"]
eunma = cx.loc[cx["단지명"].eq("은마") & cx["시군구명"].eq("강남구"), "단지일련번호"].iloc[0]
ex = similar[similar["단지일련번호"].eq(eunma)].merge(tags_latest[["태그"]], left_on="유사단지일련번호", right_index=True)
ex = ex.assign(유사단지명=ex["유사단지일련번호"].map(name_of), 시군구=ex["유사단지일련번호"].map(cx.set_index("단지일련번호")["시군구명"]))
print("은마:", tags_latest.loc[eunma, "태그"])
display(ex[["범위", "순위", "유사단지명", "시군구", "태그"]])

은마: 대단지 · 재건축 연한 도달 · 재건축 추진 중 · 중형 위주 · 역 출입구 250m 이내 · 학원가 인접


,범위,순위,유사단지명,시군구,태그
6456,서울 전체,1,한보미도맨션1,강남구,대단지 · 재건축 연한 도달 · 재건축 추진 중 · 대형 위주 · 역 출입구 500...
14327,서울 전체,2,한보미도맨션2,강남구,대단지 · 재건축 연한 도달 · 재건축 추진 중 · 대형 위주 · 역 출입구 500...
22198,서울 전체,3,"대치우성아파트1동,2동,3동,5동,6동,7동",강남구,재건축 연한 도달 · 재건축 추진 중 · 대형 위주 · 역 출입구 500m 이내 ·...
30069,서울 전체,4,대치현대,강남구,중형 위주 · 학원가 인접
37940,서울 전체,5,대치삼성,강남구,대형 위주 · 역 출입구 500m 이내 · 학원가 인접
45811,같은 구,1,한보미도맨션1,강남구,대단지 · 재건축 연한 도달 · 재건축 추진 중 · 대형 위주 · 역 출입구 500...
53682,같은 구,2,한보미도맨션2,강남구,대단지 · 재건축 연한 도달 · 재건축 추진 중 · 대형 위주 · 역 출입구 500...
61553,같은 구,3,"대치우성아파트1동,2동,3동,5동,6동,7동",강남구,재건축 연한 도달 · 재건축 추진 중 · 대형 위주 · 역 출입구 500m 이내 ·...
69424,같은 구,4,대치현대,강남구,중형 위주 · 학원가 인접
77295,같은 구,5,대치삼성,강남구,대형 위주 · 역 출입구 500m 이내 · 학원가 인접


## 7. 저장

In [13]:
for stale in ["axis_housing.joblib", "model_comparison.csv"]:  # 첫 버전 산출물 (주택특성 KMeans 폐기, 전체 단지 기준 비교표)
    (MODEL_DIR / stale).unlink(missing_ok=True)
for key in AXES:
    joblib.dump(axes[key], MODEL_DIR / f"axis_{key}.joblib")

tag_out = pd.concat({"학습기준": tags_train, "최신": tags_latest}, axis=1)
tag_out.columns = [f"{col}_{snap}" for snap, col in tag_out.columns]
tag_out = cx.set_index("단지일련번호")[["단지명", "시군구명", "법정동", "학습단지", "주요단지", "주택특성_보정"]].join(tag_out).reset_index()
tag_out.to_parquet(MODEL_DIR / "complex_tags.parquet", index=False)

keep = (["단지일련번호", "단지명", "시군구명", "학습단지", "주요단지"]
        + [f"{k}_{kind}_{tag}" for k in AXES for kind in ["cluster", "유형"] for tag in ["학습기준", "최신"]])
cx[keep].to_parquet(MODEL_DIR / "complex_clusters.parquet", index=False)
trade_clusters.to_parquet(MODEL_DIR / "trade_clusters.parquet", index=False)
similar.to_parquet(MODEL_DIR / "similar_complexes.parquet", index=False)
pd.concat({AXES[k]["label"]: profiles[k] for k in AXES}).to_csv(MODEL_DIR / "cluster_profile.csv", encoding="utf-8-sig")
(MODEL_DIR / "tag_rules.json").write_text(json.dumps({"rules": TAG_RULES, "rebuild_stage_names": REBUILD_STAGE_NAMES,
                                                      "snapshot_year": SNAP_YEAR}, ensure_ascii=False, indent=2), encoding="utf-8")

config = {
    "version": "v2",
    "train_complexes": "첫거래월 < 2026-01 (7,581)", "main_complexes": f"세대수 >= {MAIN_MIN_HOUSEHOLDS}",
    "train_snapshot": "v2_complex _학습기준 (2025-07-01)", "display_snapshot": "v2_complex _최신 (2026-07-01)",
    "price_used": False,
    "user_tags": "규칙 태그 (tag_rules.json, complex_tags.parquet)",
    "analysis_clusters": {k: {"label": AXES[k]["label"], "k": AXES[k]["k"], "features": AXES[k]["cols"],
                              "names": {int(c): n for c, n in axes[k]["names"].items()},
                              "silhouette": round(float(silhouette_score(axis_matrix(axes[k], train), labels_train[k])), 4)} for k in AXES},
    "housing_kmeans": "폐기 (주요 단지 Silhouette 0.1 안팎, housing_cluster_diagnosis.csv)",
    "knn": {"neighbors": K_NEIGHBORS, "default_scope": "같은 구", "alternative_scope": "서울 전체 (조건 유사, 가격 다를 수 있음)",
            "space": "입지 · 주택 · 교육 Feature 축별 표준화 / sqrt(축 Feature 수)"},
    "regression_handoff": "trade_clusters.parquet (입지 · 교육 KMeans, 거래 시점 배정) → 11 Ablation 후보",
}
(MODEL_DIR / "cluster_config.json").write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
print(sorted(p.name for p in MODEL_DIR.iterdir()))

['axis_education.joblib', 'axis_location.joblib', 'cluster_config.json', 'cluster_profile.csv', 'complex_clusters.parquet', 'complex_tags.parquet', 'housing_cluster_diagnosis.csv', 'knn_evaluation.csv', 'rule_vs_kmeans_comparison.csv', 'similar_complexes.parquet', 'tag_rules.json', 'trade_clusters.parquet']


## 8. 요약

| 항목 | 결정 | 근거 |
|---|---|---|
| 웹 단지 태그 | **규칙 태그 13종** (규모 · 연식 · 재건축 · 평형 · 교통 · 교육) | 법령 · 조례 · 공식 기준 · 관행을 출처와 함께 명시, 재학습과 무관 |
| 주택특성 KMeans | 폐기 | 주요 단지에서 구분력 없음 (현재 방식 Silhouette 0.1 안팎, 분리 · 가중도 0.25 안팎) |
| 입지 · 교육 KMeans | 분석 · 11 Ablation용 유지 | 학습 방식에 따른 차이 작음, 거래 시점 배정 (`trade_clusters.parquet`) |
| 유사 단지 | 같은 구 KNN(기본) + 서울 전체 KNN(대안) | 같은 구는 가격 근접, 서울 전체는 태그 일치율로 각각 검증 |

**11 회귀로 넘기는 것**

- `trade_clusters.parquet`: 입지 · 교육 군집 라벨 (거래 시점) → Ablation 후보. 주택 특성은 원본 Feature로 학습한다.
- 오차 분석은 규칙 태그(규모 · 연식 · 평형)별로 한다. 해석이 쉽고, 등급 경계(오차 범위)를 태그별로 잡을 수 있다.

**남은 한계**

- 교통 태그는 출입구 직선거리 측정값이다. 공식 역세권 판정(승강장 경계 ↔ 부지 경계, 사업지별 심의)은 공개 데이터로 재현할 수 없어 "역세권"이라는 이름을 쓰지 않는다.
  걷는 거리(보행 경로)는 웹 단계에서 실시간 경로 조회로 보완할 수 있다.
- 관행 기준(대단지 1,000세대, 신축 5년, 초등학교 300m, 학원가 1km)은 법적 정의가 없다 → 웹에서 실제 값을 함께 보여주고,
  필터에서는 사용자가 기준값을 바꿀 수 있게 한다 (`tag_rules.json`의 `웹_기준값_조정`).
- 재건축 연한 판정은 철근콘크리트 등 구조 기준이다. 층수를 거래 최고층으로 채운 단지 중 1982~1990년 준공만 영향을 받는다 (위 셀의 수).
- 05b 결측 단지(약 1.4%)는 같은 단지 거래 기록으로 물리 속성을 채웠다 (`주택특성_보정`).